# 00 — Feasibility checks

Answers the five questions that decide the research design (see the proposal), before any modelling:

1. Is `FILE_NUMBER` shared by all filings of one offering, and does it change between two campaigns of the same issuer?
2. Share of offerings (Form C) with at least one progress update (C-U).
3. What do `PROGRESSUPDATE` texts contain: an amount, a percentage, free text?
4. Share of offerings followed by an annual report (C-AR).
5. Agreement between C-U success and C-AR filing.

Decision rule: if ≥ ~70 % of offerings have a usable C-U, campaign success stays the main target. Otherwise the later Form D raise becomes the main target and success a secondary one, estimated on the reliable subsample.

**Input:** `data/raw/` filled by `python -m src.download`. **Runtime:** ~1–2 min, < 4 GB RAM.

In [1]:
import sys
from pathlib import Path

# Repo root: the folder holding src/, searched upwards from the working
# directory, then one level down (kernels do not always start in notebooks/).
here = Path.cwd().resolve()
candidates = [here, *here.parents, *(p for p in here.iterdir() if p.is_dir())]
ROOT = next((p for p in candidates if (p / "src" / "cf_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"Repo root (folder with src/cf_data.py) not found from {here}")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import cf_data as cf
from src.formd_data import filings_by_cik
from src.targets import add_annual_reports, add_formd, build_offerings, parse_amount, parse_percent, parse_outcome

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 220)

In [2]:
sub = cf.read_table("SUBMISSION")
dis = cf.read_table("DISCLOSURE")
iss = cf.read_table("ISSUER_INFORMATION")
print(f"{len(sub):,} filings, quarters {sub.QUARTER.min()} to {sub.QUARTER.max()}")
sub.SUBMISSION_TYPE.value_counts()

36,478 filings, quarters 2016q2 to 2026q3


SUBMISSION_TYPE
C/A       11823
C         11123
C-U        5668
C-AR       5370
C-W        1095
C-TR       1004
C-AR/A      294
C/A-W        34
C-TR-W       30
C-U-W        23
C-AR-W       14
Name: count, dtype: int64

## 1. `FILE_NUMBER` as the offering key

In [3]:
c = sub[sub.SUBMISSION_TYPE == "C"]
print(f"Form C filings: {len(c):,}; distinct FILE_NUMBER: {c.FILE_NUMBER.nunique():,}; distinct issuers (CIK): {c.CIK.nunique():,}")
print("Form C per FILE_NUMBER:", c.groupby("FILE_NUMBER").size().value_counts().to_dict())
print("Issuers (CIK) per FILE_NUMBER, all filings:", sub.groupby("FILE_NUMBER").CIK.nunique().value_counts().to_dict())
multi = c.groupby("CIK").filter(lambda g: len(g) > 1)
changes = (multi.groupby("CIK").FILE_NUMBER.nunique() > 1).mean()
print(f"Issuers with several Form C: {multi.CIK.nunique():,}; share whose FILE_NUMBER changes between campaigns: {changes:.1%}")
types = sub.groupby("FILE_NUMBER").SUBMISSION_TYPE.agg(set)
pd.Series({t: types.map(lambda s: t in s).mean() for t in ["C/A", "C-U", "C-AR", "C-TR", "C-W"]}, name="share of FILE_NUMBERs with this type").round(3)

Form C filings: 11,123; distinct FILE_NUMBER: 11,122; distinct issuers (CIK): 9,168
Form C per FILE_NUMBER: {1: 11121, 2: 1}
Issuers (CIK) per FILE_NUMBER, all filings: {1: 11115, 2: 6, 0: 1}


Issuers with several Form C: 1,271; share whose FILE_NUMBER changes between campaigns: 100.0%


C/A     0.492
C-U     0.447
C-AR    0.273
C-TR    0.087
C-W     0.098
Name: share of FILE_NUMBERs with this type, dtype: float64

## 2. Share of offerings with a C-U, and the success target built from it

In [4]:
off = build_offerings(sub, dis, iss)
off = add_annual_reports(off, sub)
print(f"Offerings: {len(off):,}")
print(f"With at least one C-U: {(off.N_CU > 0).mean():.1%}")
print(f"With a usable C-U (success or final failure): {off.Y_SOURCE.isin(['C-U success', 'C-U final failure']).mean():.1%}")
print(f"Classified incl. withdrawals as failures: {off.Y_SUCCESS.notna().mean():.1%}; success rate among classified: {off.Y_SUCCESS.mean():.1%}")
off.Y_SOURCE.value_counts(dropna=False)

Offerings: 11,122
With at least one C-U: 44.7%
With a usable C-U (success or final failure): 43.9%
Classified incl. withdrawals as failures: 53.3%; success rate among classified: 77.0%


Y_SOURCE
<NA>                 5198
C-U success          4559
withdrawn            1041
C-U final failure     324
Name: count, dtype: Int64

In [5]:
by_year = off.groupby(off.C_DATE.dt.year).agg(
    offerings=("Y_SUCCESS", "size"),
    with_cu=("N_CU", lambda s: (s > 0).mean()),
    classified=("Y_SUCCESS", lambda s: s.notna().mean()),
    success_rate=("Y_SUCCESS", "mean"),
)
by_year.round(3)

,offerings,with_cu,classified,success_rate
C_DATE,,,,
2016,192,0.333,0.438,0.726
2017,524,0.538,0.588,0.821
2018,763,0.439,0.514,0.824
2019,715,0.456,0.572,0.765
2020,1165,0.459,0.573,0.772
2021,1586,0.487,0.604,0.780
2022,1619,0.441,0.568,0.745
2023,1477,0.456,0.536,0.782
2024,1441,0.437,0.489,0.756


## 3. Content of `PROGRESSUPDATE` (30 random C-U)

In [6]:
cu = iss.merge(sub[["ACCESSION_NUMBER", "SUBMISSION_TYPE"]], on="ACCESSION_NUMBER")
cu = cu[cu.SUBMISSION_TYPE == "C-U"]
sample = cu.PROGRESSUPDATE.sample(30, random_state=42)
pd.DataFrame({"text": sample, "amount": sample.map(parse_amount), "percent": sample.map(parse_percent), "words": sample.map(parse_outcome)})

,text,amount,percent,words
23161,"At the close of the offering, the issuer closed on $50,000. The funds are currently being processed. The actual amount may be lower.",50000.00,NaN,NaN
34570,"The issuer has completed its Regulation Crowdfunding offering. The total amount sold in the offering was $123,600.",123600.00,NaN,NaN
13671,"Total raised $52,000",52000.00,NaN,NaN
17563,"At the close of the offering, the issuer closed on 547,559.96 and 353,685 number of securities.",547559.96,NaN,NaN
29123,"Total Amount Raised $25,611.74",25611.74,NaN,NaN
20348,"Adjustment to the initial Form C-U filing. At the close of the offering, the issuer closed on $939,296.85 and 491,447 number of securities.",939296.85,NaN,NaN
17165,"At the close of the offering, the issuer closed on 79,529.40 and 20,482 securities.",79529.40,NaN,NaN
31700,"The Offering closed successfully on 4/25/2025, having received investment commitments of $223,232.83, although a final accounting has not taken place.",223232.83,NaN,1.0
26074,"The Offering closed successfully on 2/1/2024, having received investment commitments of $226,792, although a final accounting has not taken place.",226792.00,NaN,1.0
30004,"The final number is $596,398.80 in investments.",596398.80,NaN,NaN


In [7]:
a, p, w = cu.PROGRESSUPDATE.map(parse_amount), cu.PROGRESSUPDATE.map(parse_percent), cu.PROGRESSUPDATE.map(parse_outcome)
print(f"C-U texts: {len(cu):,}; length max {cu.PROGRESSUPDATE.str.len().max()} chars")
print(f"amount found: {a.notna().mean():.1%}; percent: {p.notna().mean():.1%}; outcome in words: {w.notna().mean():.1%}; any: {(a.notna() | p.notna() | w.notna()).mean():.1%}")
cu.PROGRESSUPDATE[a.isna() & p.isna() & w.isna()].head(15)

C-U texts: 5,668; length max 256 chars
amount found: 94.1%; percent: 3.0%; outcome in words: 17.1%; any: 98.4%


230                                                     End of offering
251                                                     End of offering
418                                                     End of offering
672                                                     End of offering
721                                                     End of offering
728                                                     End of offering
734                                                     End of offering
791                                                     End of offering
800                                                     End of offering
951                                                     End of offering
1059                                                    End of offering
1072    The close date of this offering is changing to October 16, 2017
1091                                                    End of offering
1204                                                    End of o

## 4. Offerings followed by a C-AR (issuer files an annual report within 18 months after the deadline)

Only offerings whose deadline is at least 18 months before the last data quarter are counted.

In [8]:
last_filing = cf.parse_dates(sub.FILING_DATE).max()
mature = off.DEADLINE.fillna(off.C_DATE + pd.Timedelta(days=120)) <= last_filing - pd.DateOffset(months=18)
print(f"Last filing in data: {last_filing.date()}; mature offerings: {mature.sum():,}")
print(f"Share followed by a C-AR: {off.AR_AFTER[mature].mean():.1%}")

Last filing in data: 2026-09-30; mature offerings: 8,925
Share followed by a C-AR: 32.4%


## 5. Agreement between C-U outcome and C-AR

In [9]:
m = off[mature]
pd.crosstab(m.Y_SOURCE.fillna("unclassified"), m.AR_AFTER, margins=True, normalize="index").round(3)

AR_AFTER,False,True
Y_SOURCE,,
C-U final failure,0.865,0.135
C-U success,0.474,0.526
unclassified,0.805,0.195
withdrawn,0.949,0.051
All,0.676,0.324


In [10]:
usable_cu = off.Y_SOURCE.isin(["C-U success", "C-U final failure"]).mean()
with_ar_proxy = (off.Y_SUCCESS.notna() | (off.Y_SUCCESS.isna() & off.AR_AFTER)).mean()
print(f"Usable C-U: {usable_cu:.1%}  |  + withdrawals: {off.Y_SUCCESS.notna().mean():.1%}  |  + C-AR proxy: {with_ar_proxy:.1%}")
print("Decision:", "keep success as main target" if usable_cu >= 0.70 else "Form D raise becomes the main target; success is secondary")

Usable C-U: 43.9%  |  + withdrawals: 53.3%  |  + C-AR proxy: 61.1%
Decision: Form D raise becomes the main target; success is secondary


## Alternative target: Form D within 24 months

Form D notices (type `D`, not amendments) by the same CIK. Filings before the Form C give a pre-campaign Reg D indicator; filings during the campaign are parallel offerings, not follow-ons.

In [11]:
formd = filings_by_cik()
print(f"Form D filings x issuers: {len(formd):,}; dates {formd.DATE.min().date()} to {formd.DATE.max().date()}")
off = add_formd(off, formd)
w = off[off.FD_WINDOW_COMPLETE]
print(f"Offerings with a complete 24-month window: {len(w):,} ({w.CIK.nunique():,} issuers)")
print(f"Form D within 24 months: {w.Y_FORMD.mean():.1%} ({w.Y_FORMD.sum():,} positives)")
print(f"Form D before the campaign: {off.FD_BEFORE.mean():.1%}; during: {off.FD_DURING.mean():.1%}")
pd.DataFrame({
    "by launch year": w.groupby(w.C_DATE.dt.year).Y_FORMD.mean(),
}).round(3).T

Form D filings x issuers: 790,868; dates 2008-01-02 to 2026-09-30
Offerings with a complete 24-month window: 8,417 (7,082 issuers)
Form D within 24 months: 8.3% (696 positives)
Form D before the campaign: 17.8%; during: 3.7%


C_DATE,2016,2017,2018,2019,2020,2021,2022,2023,2024
by launch year,0.099,0.101,0.106,0.081,0.118,0.073,0.07,0.067,0.057


In [12]:
w.groupby(w.Y_SOURCE.fillna("unclassified")).Y_FORMD.agg(["size", "mean"]).round(3)

,size,mean
Y_SOURCE,,
C-U final failure,146,0.068
C-U success,3700,0.119
unclassified,3658,0.059
withdrawn,913,0.033


## Numbers for the proposal

In [13]:
fc = dis.merge(sub[["ACCESSION_NUMBER", "SUBMISSION_TYPE"]], on="ACCESSION_NUMBER").query("SUBMISSION_TYPE == 'C'")
rev = pd.to_numeric(fc.REVENUEMOSTRECENTFISCALYEAR, errors="coerce")
target = pd.to_numeric(fc.OFFERINGAMOUNT, errors="coerce")
print(f"Offering statements (Form C): {len(fc):,}")
print(f"Zero revenue in most recent fiscal year: {(rev == 0).mean():.1%}")
print(f"Median target: ${target.median():,.0f}; most common targets:")
target.value_counts().head(5)

Offering statements (Form C): 11,123
Zero revenue in most recent fiscal year: 44.3%
Median target: $35,000; most common targets:


OFFERINGAMOUNT
50000.0     2581
25000.0     1611
10000.0     1585
100000.0     645
20000.0      262
Name: count, dtype: int64

## Sample for manual verification on EDGAR

50 classified offerings, stratified by rule, written to `data/processed/manual_check_sample.csv`. Fill the column `MANUAL_SUCCESS` by hand from the EDGAR filings. The file is written only once, so re-running never erases the manual work.

In [14]:
rng = np.random.default_rng(0)
classified = off[off.Y_SUCCESS.notna()].reset_index()
parts = [g.sample(min(len(g), n), random_state=0) for (src, g), n in
         zip(classified.groupby("Y_SOURCE"), [10, 30, 10])]
check = pd.concat(parts)[["FILE_NUMBER", "CIK", "C_DATE", "DEADLINE", "TARGET", "AMOUNT_RAISED", "Y_SOURCE", "Y_SUCCESS", "AR_AFTER"]]
check["EDGAR"] = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=" + check.CIK
check["MANUAL_SUCCESS"] = ""
out = ROOT / "data" / "processed" / "manual_check_sample.csv"
if out.exists():  # never overwrite the hand-filled MANUAL_SUCCESS column
    print(f"{out.relative_to(ROOT)} already exists, left unchanged")
else:
    check.to_csv(out, index=False)
    print(out.relative_to(ROOT), len(check), check.Y_SOURCE.value_counts().to_dict())

data/processed/manual_check_sample.csv already exists, left unchanged
